# 🎵 Dhun — Lyrics se Gaana (free Google Colab GPU)

Yeh notebook aapka **Dhun app** asli AI engine (**ACE-Step 1.5**) ke saath chalata hai. Lyrics likho, style chuno, aur gaana bano — awaaz ke saath.

## Kaise chalayein
1. Upar menu: **Runtime → Change runtime type → T4 GPU → Save**
2. Phir **Runtime → Run all** dabao
3. Pehli baar **15–25 minute** lagenge (AI model download hota hai). Sabse neeche wale cell mein ek **link** aayega — use kholo, wahi aapka Dhun app hai.
4. Is tab ko **khula rakho**. Tab band ya Colab band = app band. Agli baar phir se *Run all*.

> Free Colab kuch ghante baad apne aap band ho jaata hai. Bane hue gaane app se **download** kar lena.

In [ ]:
#@title 1️⃣ GPU check + ACE-Step install (5–10 min)
import subprocess, os
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("❌ GPU nahi mila. Runtime → Change runtime type → T4 GPU chuno, phir Run all.")
print("✅ GPU:", gpu)

os.chdir("/content")
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
if not os.path.isdir("/content/ACE-Step-1.5"):
    !git clone -q --depth 1 https://github.com/ACE-Step/ACE-Step-1.5.git
%cd /content/ACE-Step-1.5
!uv sync 2>&1 | tail -n 3
print("✅ ACE-Step install ho gaya")

In [ ]:
#@title 2️⃣ AI engine start karo (pehli baar model download: 10–15 min)
import subprocess, time, urllib.request, os
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
# Colab sets MPLBACKEND to its notebook backend, which crashes matplotlib inside ACE-Step's own venv
env = dict(os.environ, MPLBACKEND="Agg")
log = open("/content/acestep.log", "w")
ace = subprocess.Popen(["uv", "run", "acestep-api", "--host", "127.0.0.1", "--port", "8001"],
                       cwd="/content/ACE-Step-1.5", env=env, stdout=log, stderr=subprocess.STDOUT)
t0, ready, failed = time.time(), False, False
while not ready and not failed:
    try:
        urllib.request.urlopen("http://127.0.0.1:8001/health", timeout=5); ready = True
    except Exception:
        failed = ace.poll() is not None
        if time.time() - t0 > 40 * 60: failed = True
        if not failed:
            print(f"⏳ AI engine load ho raha hai… {int(time.time()-t0)//60} min", end="\r"); time.sleep(10)
if failed:
    print(open("/content/acestep.log").read()[-3000:])
    print("\n❌ AI engine ruk gaya — upar ka log screenshot karke bhejo.")
else:
    print(f"\n✅ AI engine ready ({int(time.time()-t0)//60} min)")

In [ ]:
#@title 3️⃣ Dhun app start karo
import subprocess, time, urllib.request, os
os.chdir("/content")
if not os.path.isdir("/content/Dhun"):
    !git clone -q --depth 1 https://github.com/digyservice-oss/Dhun.git
else:
    !cd /content/Dhun && git pull -q
!pip -q install flask numpy scipy pillow
env = dict(os.environ, ENGINE="ace", ACESTEP_URL="http://127.0.0.1:8001", DAILY_LIMIT="0",
           MAX_DURATION="300", DATA_DIR="/content/dhun-data", PORT="7860")
app = subprocess.Popen(["python", "app.py"], cwd="/content/Dhun", env=env,
                       stdout=open("/content/dhun.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try: urllib.request.urlopen("http://127.0.0.1:7860/", timeout=3); break
    except Exception: time.sleep(2)
else:
    print(open("/content/dhun.log").read()[-3000:]); raise SystemExit("❌ Dhun app start nahi hua.")
print("✅ Dhun app chal raha hai")

In [ ]:
#@title 4️⃣ 🔗 Aapka link (yahan click karo)
import subprocess, re, time, os
if not os.path.exists("/content/cloudflared"):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tlog = "/content/tunnel.log"
tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:7860"],
                       stdout=open(tlog, "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(tlog).read())
    if m: url = m.group(0); break
    time.sleep(1)
if not url: raise SystemExit("❌ Link nahi bana. Yeh cell dobara chalao.")
from IPython.display import HTML, display
display(HTML(f'<h2>🎵 Aapka Dhun app: <a href="{url}" target="_blank">{url}</a></h2>'
             '<p>Link kholo → lyrics likho → gaana banao. Yeh Colab tab khula rakhna.</p>'
             '<p>Pehla gaana banne mein thoda zyada time lag sakta hai.</p>'))

### Dikkat aaye to
- **Link nahi khul raha:** cell 4 dobara chalao, naya link milega.
- **Gaana fail:** `Runtime → Restart session` phir `Run all`.
- **Log dekhna ho:** neeche wala cell chalao.

In [ ]:
#@title 🛠 Logs (sirf dikkat aaye tab)
!tail -n 30 /content/acestep.log; echo ----; tail -n 30 /content/dhun.log